# Case Study Project: Unlocking Content Intelligence, A Deep Dive into Netflix’s Global Library

**Client Brief:**
Netflix's Content Strategy & Analytics Team wants to make data-driven decisions on licensing,
regional content expansion, and user personalization. As a Junior Data Analyst intern, you have been
asked to prepare a detailed exploratory report using their internal dataset of titles.
The leadership team is particularly interested in understanding patterns in the catalog and how those
might inform strategic initiatives, such as producing local-language content or acquiring longer or
shorter titles.

__Your Role:__
You will **explore**, **clean**, **transform**, and **analyze** the data using advanced exploratory techniques. Your
report should include visual insights, statistical summaries, feature insights, and practical takeaways,
all using **Python, Pandas, NumPy, and Seaborn/Matplotlib**.


**Download csv data**
[Download csv data](https://github.com/prasertcbs/basic-dataset/blob/master/netflix_titles.csv)

## Section 1: Data Familiarization and Cleaning
### 1. Load the Netflix dataset and display basic info (shape, columns, datatypes).

In [1]:
import pandas as pd
import numpy as np

# Reference year for "content age". A fixed constant keeps the notebook reproducible
# (datetime.now() would silently change the numbers every year).
REFERENCE_YEAR = 2025

In [2]:
df = pd.read_csv('../../data/netflix_titles_homework.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,81145628,Movie,Norm of the North: King Sized Adventure,"Richard Finn, Tim Maltby","Alan Marriott, Andrew Toth, Brian Dobson, Cole...","United States, India, South Korea, China","September 9, 2019",2019,TV-PG,90 min,"Children & Family Movies, Comedies",Before planning an awesome wedding for his gra...
1,80117401,Movie,Jandino: Whatever it Takes,NaN,Jandino Asporaat,United Kingdom,"September 9, 2016",2016,TV-MA,94 min,Stand-Up Comedy,Jandino Asporaat riffs on the challenges of ra...
2,70234439,TV Show,Transformers Prime,NaN,"Peter Cullen, Sumalee Montano, Frank Welker, J...",United States,"September 8, 2018",2013,TV-Y7-FV,1 Season,Kids' TV,"With the help of three human allies, the Autob..."
3,80058654,TV Show,Transformers: Robots in Disguise,NaN,"Will Friedle, Darren Criss, Constance Zimmer, ...",United States,"September 8, 2018",2016,TV-Y7,1 Season,Kids' TV,When a prison ship crash unleashes hundreds of...
4,80125979,Movie,#realityhigh,Fernando Lebrija,"Nesta Cooper, Kate Walsh, John Michael Higgins...",United States,"September 8, 2017",2017,TV-14,99 min,Comedies,When nerdy high schooler Dani finally attracts...


In [3]:
df.shape

(6234, 12)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6234 entries, 0 to 6233
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       6234 non-null   int64
 1   type          6234 non-null   str  
 2   title         6234 non-null   str  
 3   director      4265 non-null   str  
 4   cast          5664 non-null   str  
 5   country       5758 non-null   str  
 6   date_added    6223 non-null   str  
 7   release_year  6234 non-null   int64
 8   rating        6224 non-null   str  
 9   duration      6234 non-null   str  
 10  listed_in     6234 non-null   str  
 11  description   6234 non-null   str  
dtypes: int64(2), str(10)
memory usage: 584.6 KB


In [5]:
df.dtypes

show_id         int64
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

In [6]:
print("duplicate rows:", df.duplicated().sum(), "| duplicate show_id:", df['show_id'].duplicated().sum())
print("release_year range:", df['release_year'].min(), "-", df['release_year'].max())
print(df['type'].value_counts().to_dict())

duplicate rows: 0 | duplicate show_id: 0
release_year range: 1925 - 2020
{'Movie': 4265, 'TV Show': 1969}


### 2. Identify and handle missing values:  
o Use dropna() for critical fields.  
o Use fillna() where appropriate (e.g., fill missing country or rating).  
o Experiment with IterativeImputer for fields with numerical implications (like
release_year).  

**Note on `IterativeImputer`:** the brief suggests it for numeric fields such as `release_year`, but `release_year` has no missing values in this file, so there is nothing to impute. Every column with missing values is categorical text (`director`, `cast`, `country`, `rating`) or a date, for which model-based numeric imputation is not appropriate, so simple, documented rules are used instead.

In [7]:
df.isnull().sum()

show_id            0
type               0
title              0
director        1969
cast             570
country          476
date_added        11
release_year       0
rating            10
duration           0
listed_in          0
description        0
dtype: int64

Missing-value policy, decided column by column:

| Column | Missing | Decision | Why |
|---|---|---|---|
| `director` | many (~32 %) | fill with `'Unknown'` | too many to drop; absence is itself informative |
| `cast` | some | fill with `'Unknown'` | same |
| `country` | some | drop rows | needed for geographic analysis |
| `date_added`, `rating` | very few | drop rows | tiny loss, both are needed downstream |

In [8]:
n_before = len(df)
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df = df.dropna(subset=['country', 'date_added', 'rating']).copy()
print(f"rows: {n_before} -> {len(df)} ({n_before - len(df)} dropped, {(n_before - len(df)) / n_before:.1%})")
print("remaining missing values:", int(df.isnull().sum().sum()))

rows: 6234 -> 5743 (491 dropped, 7.9%)
remaining missing values: 0


### 3. Parse and clean columns:  
o Extract numeric values from duration.  
o Convert date_added to datetime format.  
o Create a new column for content age (current year – release year).  

In [9]:
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed')
df['year_added'] = df['date_added'].dt.year
df.dtypes

show_id                  int64
type                       str
title                      str
director                   str
cast                       str
country                    str
date_added      datetime64[us]
release_year             int64
rating                     str
duration                   str
listed_in                  str
description                str
year_added               int32
dtype: object

Content age is `REFERENCE_YEAR - release_year`. `duration` mixes two units: minutes for movies (`"90 min"`) and seasons for TV shows (`"2 Seasons"`). Each is extracted into its own column and left **missing (NaN) where it does not apply**. Filling those with 0 would make a TV show a "0-minute title" and drag every average duration down.

In [10]:
df['content_age'] = REFERENCE_YEAR - df['release_year']
df['duration_min'] = df['duration'].str.extract(r'(\d+)\s*min', expand=False).astype(float)
df['seasons'] = df['duration'].str.extract(r'(\d+)\s*Season', expand=False).astype(float)

# Every title should have exactly one of the two units, and it must match its type
assert (df['duration_min'].notna() ^ df['seasons'].notna()).all(), "a title has neither or both duration units"
assert (df.loc[df.type == 'Movie', 'duration_min'].notna()).all()
assert (df.loc[df.type == 'TV Show', 'seasons'].notna()).all()
df[['type', 'duration', 'duration_min', 'seasons', 'content_age']].head()

,type,duration,duration_min,seasons,content_age
0,Movie,90 min,90.0,NaN,6
1,Movie,94 min,94.0,NaN,9
2,TV Show,1 Season,NaN,1.0,12
3,TV Show,1 Season,NaN,1.0,9
4,Movie,99 min,99.0,NaN,8


Release eras. The bins are right-inclusive, so `(1900, 1980]` is "Classic Era", `(1980, 2000]` "Late 20th Century", and so on; the check below confirms nothing falls outside the bins.

In [11]:
df['era'] = pd.cut(
    df['release_year'],
    bins=[1900, 1980, 2000, 2015, float('inf')],
    labels=['Classic Era', 'Late 20th Century', 'Early 21st Century', 'Modern Era']
)
assert df['era'].notna().all(), "release_year outside the era bins"
df['era'].value_counts().sort_index()

era
Classic Era            104
Late 20th Century      296
Early 21st Century    1981
Modern Era            3362
Name: count, dtype: int64

`is_new` flags titles released in 2020 or later. In this file very few titles qualify, because the dataset only covers titles added up to early 2020, so this flag is rarely true and should be interpreted with that in mind.

In [12]:
df['is_new'] = (df['release_year'] >= 2020).astype(int)
print(df['is_new'].value_counts().to_string())
print("latest date_added in the file:", df['date_added'].max().date())

is_new
0    5725
1      18
latest date_added in the file: 2020-01-18


## Cleaned data summary

A final integrity check on the cleaned table, and a look at the columns created in this section.

In [13]:
not_applicable = ['duration_min', 'seasons']   # NaN by design (unit does not apply to the title type)
assert df.drop(columns=not_applicable).isnull().sum().sum() == 0
assert df['show_id'].is_unique
print(f"cleaned shape: {df.shape}")
df[['title', 'type', 'country', 'date_added', 'release_year', 'content_age', 'duration_min', 'seasons', 'era', 'is_new']].head()

cleaned shape: (5743, 18)


,title,type,country,date_added,release_year,content_age,duration_min,seasons,era,is_new
0,Norm of the North: King Sized Adventure,Movie,"United States, India, South Korea, China",2019-09-09,2019,6,90.0,NaN,Modern Era,0
1,Jandino: Whatever it Takes,Movie,United Kingdom,2016-09-09,2016,9,94.0,NaN,Modern Era,0
2,Transformers Prime,TV Show,United States,2018-09-08,2013,12,NaN,1.0,Early 21st Century,0
3,Transformers: Robots in Disguise,TV Show,United States,2018-09-08,2016,9,NaN,1.0,Modern Era,0
4,#realityhigh,Movie,United States,2017-09-08,2017,8,99.0,NaN,Modern Era,0


In [14]:
# Movies and TV shows summarised separately: duration units differ
df.groupby('type').agg(titles=('show_id', 'size'),
                       median_minutes=('duration_min', 'median'),
                       median_seasons=('seasons', 'median'),
                       median_content_age=('content_age', 'median')).round(1)

,titles,median_minutes,median_seasons,median_content_age
type,,,,
Movie,4066,98.0,NaN,9.0
TV Show,1677,NaN,1.0,8.0


## Methodology notes

- **Missing data:** every decision is explicit and counted (rows lost reported above); nothing is filled silently.
- **Not-applicable vs zero:** `duration_min` is NaN for TV shows and `seasons` is NaN for movies, so group statistics are not diluted by fake zeros.
- **Reproducibility:** `content_age` uses a fixed `REFERENCE_YEAR`; `is_new` has a column name safe for attribute access.
- **Checks:** assertions confirm the duration parse, the era bins and that no unexpected nulls remain, so a changed input file fails loudly.
- **Scope:** this notebook covers Section 1 (load, clean, transform) of the brief only.